# RSNA Knee — soumission de la baseline quatre lecteurs

**Inférence uniquement** : quatre encodeurs génériques gelés, vingt têtes déjà entraînées,
moyenne des cinq folds puis mélange de rangs `candidate05` (5 % DINOv2).
Ce notebook relit le test officiel à chaque exécution, y compris lors du scoring caché.
Il produit `/kaggle/working/submission.csv`. Aucun entraînement n'est relancé.

À joindre : le **nouveau Dataset** créé depuis `UPLOAD_SUBMISSION_TO_KAGGLE.zip`
et la compétition **RSNA Knee Abnormality Detection**. Activer **GPU T4 x2** et
désactiver **Internet**. Lire `GUIDE_SOUMISSION_FR.md`.

Le notebook s'ouvre en inspection. Passer `RUN_INFERENCE` à `True` pour exécuter.

## 1. Paramètres

**Objectif :** autoriser explicitement l'inférence et définir les chemins.
**Prérequis :** paquet de soumission et compétition attachés.
**Sortie :** paramètres en mémoire. Les chemins vides sont détectés automatiquement.
Garder les poids de mélange prédéfinis ; ne pas choisir une variante sur les mêmes 58 gold.

In [ ]:
RUN_INFERENCE = False  # Passer à True avant Save Version → Save & Run All.
BUNDLE_DIR = ""         # Dataset contenant submission_bundle.json.
COMPETITION_DIR = ""    # Dossier officiel avec test.csv et test_series/.
OUTPUT_DIR = "/kaggle/working"
WORK_DIR = "/kaggle/temp/rsna_submission"  # Codecs et petites images temporaires.

## 2. Installation hors ligne et vérification

**Objectif :** vérifier les poids/têtes et installer les wheels incluses.
**Prérequis :** GPU NVIDIA et Internet désactivé ; exécuter avant tout import de timm/pydicom/cv2.
**Sortie :** dépendances hors ligne et chemins vérifiés. Torch/torchvision CUDA du service
restent en place. Aucun DICOM n'est encore traité.

In [ ]:
if RUN_INFERENCE:
    from pathlib import Path
    import hashlib, importlib.util, json, os, sys
    import torch

    if sys.version_info < (3, 11) or not torch.cuda.is_available():
        raise RuntimeError("Utiliser une session Kaggle Python >= 3.11 avec GPU NVIDIA activé.")
    sys.dont_write_bytecode = True
    if BUNDLE_DIR:
        found = [Path(BUNDLE_DIR)]
    else:
        found = []
        for directory, subdirs, files in os.walk("/kaggle/input"):
            path = Path(directory)
            subdirs[:] = [x for x in subdirs if x not in {
                "competitions", "rsna-knee-abnormality-detection", "train_series", "test_series"}]
            if len(path.relative_to("/kaggle/input").parts) >= 5:
                subdirs[:] = []
            if "submission_bundle.json" in files:
                found.append(path)
    if len(found) != 1:
        raise FileNotFoundError("Ajouter le nouveau Dataset de soumission ou renseigner BUNDLE_DIR.")
    bundle = found[0]
    metadata = json.loads((bundle / "submission_bundle.json").read_text())
    bootstrap_path = bundle / "code/rsna_submit_bootstrap.py"
    with bootstrap_path.open("rb") as stream:
        bootstrap_sha = hashlib.file_digest(stream, "sha256").hexdigest()
    if bootstrap_sha != metadata["files"]["code/rsna_submit_bootstrap.py"]["sha256"]:
        raise ValueError("Code d'installation altéré ; vérifier le Dataset.")
    specification = importlib.util.spec_from_file_location("rsna_submit_bootstrap", bootstrap_path)
    bootstrap = importlib.util.module_from_spec(specification)
    specification.loader.exec_module(bootstrap)
    manifest = bootstrap.install(bundle, WORK_DIR)
    competition = bootstrap.find_competition(explicit=COMPETITION_DIR)
    print("GPU :", torch.cuda.get_device_name(0))
    print("Paquet :", bundle)
    print("Compétition :", competition)
else:
    print("Mode inspection : passer RUN_INFERENCE à True pour lancer la soumission.")

## 3. Prédire toutes les études du test courant

**Objectif :** lire les DICOM, sélectionner les six slots et huit images maximum/slot,
extraire les quatre représentations puis appliquer les vingt têtes finales.
**Prérequis :** installation réussie et DICOM officiels accessibles.
**Sorties :** `submission.csv`, `inference_report.json`, diagnostics et prédictions intermédiaires.
La conversion min–max/resize/JPEG est empirique et auditée ; les JPEG d'origine ne
fournissent pas de fenêtres anatomiques 2.5D. Les rangs sont calculés sur TOUT le test,
après la moyenne des cinq probabilités de fold, jamais séparément par lot.

In [ ]:
if RUN_INFERENCE:
    from knee_submit.inference import run_inference
    submission, report = run_inference(
        bundle=bundle, competition=competition, output=OUTPUT_DIR,
        device="cuda", input_format="dicom", work_dir=WORK_DIR)
    print(submission.head().to_string(index=False))
    print("Études traitées :", report["studies"])
    print("Têtes utilisées :", report["head_count"])

## 4. Contrôler les sorties avant soumission

**Objectif :** vérifier le CSV final et consulter la durée/les erreurs de décodage.
**Prérequis :** inférence entièrement terminée.
**Sortie :** résumé lisible. Une erreur pendant l'inférence ne laisse pas un ancien
`submission.csv`. Les trois exemples visibles ne constituent pas un score leaderboard.

In [ ]:
if RUN_INFERENCE:
    import json
    from pathlib import Path
    from knee_submit.blend import validate_submission
    from knee_submit.data import read_inputs
    ids, _ = read_inputs(competition)
    validate_submission(submission, ids)
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print("Fichier à sélectionner :", Path(OUTPUT_DIR) / "submission.csv")

## 5. Envoyer la version Kaggle

1. Vérifier `RUN_INFERENCE = True`, GPU activé, Internet désactivé et les deux inputs joints.
2. Utiliser **Save Version → Save & Run All**. Attendre la fin réussie.
3. Ouvrir cette version terminée, puis **Submit to Competition** ; sélectionner **submission.csv**.
4. Kaggle réexécute le notebook sur le test caché. Le nombre de lignes suivra son nouveau test.csv.

Ne pas soumettre directement un CSV local de trois lignes. Ce notebook emploie le
candidat principal fixé à l'avance et ne réutilise ni les anciennes prédictions test
ni les caractéristiques train. Conserver le Dataset des poids attaché à la version.